We will join credits csv with audio feats csv so that credits gets the spotify id.

Currently the only identifier on credits is the track name, which isn't the strongest unique string, though it should still work fine. However there can always be little discrepancies with track names here and there in different sources. Spotify's song id should, hopefully, remain consistent.

In [6]:
import os
import json
import pandas as pd

In [4]:
with open('consts.json', 'r') as file:
    data = json.load(file)

album = data['album']
artist = data['artist']

In [7]:
script_dir = os.path.dirname(os.path.abspath('join-data.ipynb'))

In [22]:
credits_csv_file_path = os.path.join(script_dir, '..', 'albums', f'{album}', 'output', f'credits_{album}.csv')
audio_feats_w_play_counts_csv_file_path = os.path.join(script_dir, '..', 'albums', f'{album}', 'output', f'audio_feats_and_play_counts_{album}.csv')

In [23]:
df_credits = pd.read_csv(credits_csv_file_path)
df_audio = pd.read_csv(audio_feats_w_play_counts_csv_file_path)

In [21]:
def normalize(s):
    return s.strip().lower().replace("’", "'")

In [24]:
df_audio["join_key"] = df_audio["track_name"].map(normalize)
df_credits["join_key"] = df_credits["track_name"].map(normalize)

In [26]:
# sanity check before trusting the join
unmatched = set(df_credits["join_key"]) - set(df_credits["join_key"])
if unmatched:
    print("Unmatched credit rows — check these manually:", unmatched)

In [28]:
credits_with_id = df_credits.merge(
    df_audio[["join_key", "spotify_track_id"]], on="join_key", how="left"
)

In [29]:
credits_with_id

,track_name,role,role_detail,person,is_primary,join_key,spotify_track_id
0,The Questions,producer,producer,J Dilla,True,the questions,12DQLP0EURlKcwguEJM5oY
1,The Questions,producer,producer,James Poyser,False,the questions,12DQLP0EURlKcwguEJM5oY
2,The Questions,writer,writer,James Poyser,True,the questions,12DQLP0EURlKcwguEJM5oY
3,The Questions,writer,writer,Common,False,the questions,12DQLP0EURlKcwguEJM5oY
4,The Questions,writer,writer,Mos Def,False,the questions,12DQLP0EURlKcwguEJM5oY
...,...,...,...,...,...,...,...
182,Dooinit,instrumentalist,lead_vocals,Common,True,dooinit,7bEEzWWgJS4HhzYtNLCXfa
183,Dooinit,recorder,recorder,Todd Fairall,True,dooinit,7bEEzWWgJS4HhzYtNLCXfa
184,Dooinit,recorder,assistant_recorder,Nick Hura,True,dooinit,7bEEzWWgJS4HhzYtNLCXfa
185,Dooinit,mixer,mixer,Bob Power,True,dooinit,7bEEzWWgJS4HhzYtNLCXfa


In [31]:
def get_primary(role_name, group):
    match = group[(group.role == role_name) & (group.is_primary == True)]
    return match["person"].iloc[0] if len(match) else None

In [33]:
records = []
for spotify_track_id, group in credits_with_id.groupby("spotify_track_id"):
    track_row = df_audio[df_audio.spotify_track_id == spotify_track_id].iloc[0]
    records.append({
        "spotify_track_id": spotify_track_id,
        "track_name": track_row["track_name"],
        "track_number": track_row.get("track_number"),
        "play_count": int(track_row["play_count"]),
        "play_count_last_updated": track_row["play_count_last_updated"],
        "audio_features": {
            "energy": track_row["energy"],
            "danceability": track_row["danceability"],
            "valence": track_row["valence"],
            "tempo": track_row["tempo"],
        },
        "primary_producer": get_primary("producer", group),
        "primary_mixer": get_primary("mixer", group),
        "credits": group[["role", "role_detail", "person", "is_primary"]].to_dict("records"),
    })

In [35]:
records[0]

{'spotify_track_id': '12DQLP0EURlKcwguEJM5oY',
 'track_name': 'The Questions',
 'track_number': np.int64(7),
 'play_count': 3180740,
 'play_count_last_updated': '2026-08-16',
 'audio_features': {'energy': np.float64(0.388),
  'danceability': np.float64(0.828),
  'valence': np.float64(0.82),
  'tempo': np.float64(90.742)},
 'primary_producer': 'J Dilla',
 'primary_mixer': 'Bob Power',
 'credits': [{'role': 'producer',
   'role_detail': 'producer',
   'person': 'J Dilla',
   'is_primary': True},
  {'role': 'producer',
   'role_detail': 'producer',
   'person': 'James Poyser',
   'is_primary': False},
  {'role': 'writer',
   'role_detail': 'writer',
   'person': 'James Poyser',
   'is_primary': True},
  {'role': 'writer',
   'role_detail': 'writer',
   'person': 'Common',
   'is_primary': False},
  {'role': 'writer',
   'role_detail': 'writer',
   'person': 'Mos Def',
   'is_primary': False},
  {'role': 'writer',
   'role_detail': 'writer',
   'person': 'J Dilla',
   'is_primary': False},

In [36]:
output_file_path = os.path.join(script_dir, '..', 'albums', f'{album}', 'output', f'album_data_viz_{album}.json')

In [37]:
import json

def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_serializable(v) for v in obj]
    if pd.isna(obj):
        return None
    if hasattr(obj, "item"):  # numpy int64/float64 -> python native
        return obj.item()
    return obj

with open(output_file_path, "w") as f:
    json.dump(make_serializable(records), f, indent=2)